# Notebook 01 — Introducción a PL/pgSQL y bloques anónimos

Primer sub-bloque del Tema 06. **PL/pgSQL** (*Procedural Language for PostgreSQL*) es el lenguaje procedural que vive **dentro del motor**. Te permite escribir lógica de control (`IF`, `CASE`, `FOR`, `WHILE`), declarar variables, manejar excepciones y empaquetar todo en funciones y procedimientos almacenados.

En este notebook conoces el lenguaje desde su forma más simple — el **bloque anónimo** `DO $$ ... $$;` — antes de pasar a las estructuras de control (Notebook 02), funciones (Notebook 03) y procedimientos (Notebook 04).

Al terminar deberías poder: ejecutar bloques anónimos, declarar variables, asignar valores, generar mensajes con `RAISE`, y entender qué hace `PL/pgSQL` que SQL puro **no puede hacer**.

**Contenido de este notebook:**

- [Setup](#setup)
- [¿Por qué PL/pgSQL?](#por-qué-plpgsql)
- [El bloque anónimo `DO $$ ... $$;`](#el-bloque-anónimo-do----)
- [Declaración y asignación de variables](#declaración-y-asignación-de-variables)
- [Mensajes con `RAISE`](#mensajes-con-raise)
- [Asignar desde una query con `INTO`](#asignar-desde-una-query-con-into)
- [`PERFORM` — ejecutar una query sin usar el resultado](#perform--ejecutar-una-query-sin-usar-el-resultado)
- [Bloques anidados y scope](#bloques-anidados-y-scope)
- [Manejo básico de excepciones](#manejo-básico-de-excepciones)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine

## ¿Por qué PL/pgSQL?

SQL puro es **declarativo**: describes qué quieres, el motor decide cómo obtenerlo. Eso es genial para la mayoría de operaciones de BI — un `SELECT` con `GROUP BY` cubre el 80% del trabajo.

Pero hay cosas que SQL puro **no puede expresar directamente**:

- Lógica condicional compleja: "si hay más de X clientes en la región Y, ejecuta el proceso Z; si no, otro distinto".
- Iteración con estado: "para cada categoría, calcula su ranking acumulado y guarda el resultado".
- Manejo de errores: "intenta cargar; si falla por FK, registra el error y continúa con la siguiente fila".
- Empaquetar lógica reutilizable: "esta serie de pasos los necesito desde 5 lugares distintos".

**PL/pgSQL agrega:**

| Capacidad | SQL puro | PL/pgSQL |
|---|---|---|
| Variables locales | ❌ | ✅ `DECLARE` |
| Condicionales | Solo `CASE` (expresión) | ✅ `IF`, `CASE` (sentencia) |
| Loops | ❌ | ✅ `LOOP`, `FOR`, `WHILE` |
| Manejo de excepciones | ❌ | ✅ `BEGIN ... EXCEPTION ... END` |
| Funciones reutilizables | Limitado | ✅ `CREATE FUNCTION` |
| Procedimientos con efectos | ❌ | ✅ `CREATE PROCEDURE` |

**La regla operativa:** quédate en SQL puro siempre que puedas. PL/pgSQL es para cuando SQL puro **realmente no alcanza**. El motor optimiza mucho mejor un `INSERT … SELECT` que un `FOR` loop fila por fila.

## El bloque anónimo `DO $$ ... $$;`

La forma más simple de ejecutar PL/pgSQL: un bloque **anónimo** que corre una vez y se descarta. Sintaxis básica:

```sql
DO $$
BEGIN
    -- código PL/pgSQL aquí
END
$$;
```

Las claves:

- **`DO`** — palabra clave que dice "voy a ejecutar un bloque procedural".
- **`$$ ... $$`** — *dollar-quoted string*. Es la forma de PostgreSQL de delimitar el código sin tener que escapar comillas internas. Todo lo que esté entre los dos `$$` es el cuerpo del bloque.
- **`BEGIN ... END`** — los delimitadores del bloque PL/pgSQL propiamente dicho.
- El `;` final cierra la sentencia `DO`.

Un primer ejemplo:

In [ ]:
%%sql
DO $$
BEGIN
    RAISE NOTICE 'Hola desde PL/pgSQL';
END
$$;

El `RAISE NOTICE` imprime un mensaje al cliente. En psql lo verías con prefijo `NOTICE:`. En JupySQL aparece como output de texto debajo de la celda.

**Sobre `$$` y delimitadores alternativos:**

Si tu código contiene un `$$` literal (raro pero pasa), usa **delimitadores con etiqueta**:

```sql
DO $body$
BEGIN
    RAISE NOTICE 'Aquí adentro puedo usar $$ sin problema';
END
$body$;
```

Las etiquetas pueden ser cualquier identificador (`$body$`, `$func$`, `$proc$`). Solo asegúrate de que la apertura y cierre coincidan.

## Declaración y asignación de variables

Las variables se declaran en una sección **`DECLARE`** que va **antes** del `BEGIN`:

```sql
DO $$
DECLARE
    nombre  TEXT;            -- declarada, valor NULL por default
    edad    INTEGER := 30;   -- declarada e inicializada
    activo  BOOLEAN := TRUE;
BEGIN
    -- código
END
$$;
```

Detalles importantes:

- **Tipos:** cualquier tipo SQL válido (`TEXT`, `NUMERIC(10,2)`, `DATE`, `BOOLEAN`, etc.) más tipos compuestos como `RECORD`, `%ROWTYPE`, `%TYPE`.
- **Asignación:** el operador es **`:=`**, NO `=`. El `=` se usa para comparación.
- **Constantes:** agrega `CONSTANT` antes del tipo: `pi CONSTANT NUMERIC := 3.14159;`.
- **NOT NULL:** puedes forzar que una variable no pueda ser NULL: `nombre TEXT NOT NULL := 'Ana';` (necesita valor inicial).

In [ ]:
%%sql
DO $$
DECLARE
    nombre   TEXT     := 'Aurora';
    version  NUMERIC  := 17.4;
    activo   BOOLEAN  := TRUE;
BEGIN
    RAISE NOTICE 'Motor: %, versión: %, activo: %', nombre, version, activo;
END
$$;

Los `%` dentro del string son **placeholders** que se reemplazan en orden con los argumentos siguientes. Es el `printf` del PL/pgSQL.

**Asignación durante la ejecución:**

In [ ]:
%%sql
DO $$
DECLARE
    contador INTEGER := 0;
BEGIN
    contador := contador + 1;        -- ahora vale 1
    contador := contador * 10;       -- ahora vale 10
    RAISE NOTICE 'Contador final: %', contador;
END
$$;

**Tipos basados en columnas existentes — `%TYPE` y `%ROWTYPE`:**

Útiles para evitar repetir tipos:

```sql
DECLARE
    nombre_cliente  northwind_dwh.dim_customer.company_name%TYPE;   -- mismo tipo que la columna
    fila_cliente    northwind_dwh.dim_customer%ROWTYPE;             -- una fila completa
```

Si después cambias el tipo de la columna, las variables se ajustan automáticamente.

## Mensajes con `RAISE`

`RAISE` es el comando para **enviar mensajes al cliente**. Tiene varios niveles:

| Nivel | Para qué | Detiene ejecución |
|---|---|---|
| `DEBUG` | Información detallada de depuración | No |
| `LOG` | Eventos significativos para auditoría | No |
| `INFO` | Información general | No |
| `NOTICE` | Notas útiles (el más común para mostrar valores) | No |
| `WARNING` | Algo no fatal pero anómalo | No |
| `EXCEPTION` | Error fatal — aborta el bloque y rollback | **Sí** |

**Sintaxis genérica:**

```sql
RAISE NIVEL 'mensaje con % y % placeholders', valor1, valor2;
```

In [ ]:
%%sql
DO $$
BEGIN
    RAISE NOTICE 'Esto es informativo';
    RAISE WARNING 'Esto es una advertencia';
    -- RAISE EXCEPTION 'Esto abortaría aquí, no llegaríamos a la siguiente línea';
    RAISE NOTICE 'Termina la ejecución';
END
$$;

**Lanzar un error explícito:**

In [ ]:
%%sql
DO $$
DECLARE
    edad INTEGER := -5;
BEGIN
    IF edad < 0 THEN
        RAISE EXCEPTION 'Edad no puede ser negativa, fue %', edad;
    END IF;
    RAISE NOTICE 'Edad válida: %', edad;
END
$$;

El bloque aborta con error explícito. En código productivo así marcas pre-condiciones violadas o estados inválidos.

## Asignar desde una query con `INTO`

Para obtener un valor de la base y guardarlo en una variable, usa **`SELECT ... INTO variable`**:

```sql
SELECT COUNT(*) INTO total_clientes FROM dim_customer;
```

**Caveat importante:** si la query devuelve más de una fila, la variable solo guarda **la primera** (no determinista sin `ORDER BY`). Si quieres garantía de fila única, usa **`SELECT ... INTO STRICT`** — falla si la query devuelve 0 o más de 1.

In [ ]:
%%sql
DO $$
DECLARE
    total_clientes INTEGER;
    cliente_top    TEXT;
    ventas_top     NUMERIC;
BEGIN
    -- Asignación simple
    SELECT COUNT(*) INTO total_clientes FROM northwind_dwh.dim_customer;
    
    -- Asignación múltiple — varias columnas en una sola query
    SELECT dc.company_name, SUM(fs.line_total)
      INTO cliente_top, ventas_top
      FROM northwind_dwh.fact_sales fs
      JOIN northwind_dwh.dim_customer dc USING (customer_key)
     GROUP BY dc.company_name
     ORDER BY 2 DESC
     LIMIT 1;
    
    RAISE NOTICE 'Total clientes: %', total_clientes;
    RAISE NOTICE 'Cliente top: % con $%', cliente_top, ROUND(ventas_top, 2);
END
$$;

**Verificar si una query devolvió algo — `FOUND`:**

Después de cualquier sentencia que modifique filas o un `SELECT INTO`, PL/pgSQL setea automáticamente la variable booleana **`FOUND`**:

In [ ]:
%%sql
DO $$
DECLARE
    company TEXT;
BEGIN
    SELECT company_name INTO company
      FROM northwind_dwh.dim_customer
     WHERE customer_id = 'XXXXX';     -- no existe
    
    IF NOT FOUND THEN
        RAISE NOTICE 'No se encontró cliente XXXXX';
    ELSE
        RAISE NOTICE 'Cliente: %', company;
    END IF;
END
$$;

## `PERFORM` — ejecutar una query sin usar el resultado

PL/pgSQL **no permite** `SELECT` sueltos cuyo resultado se descarte — el motor te exige que hagas algo con el output. Si quieres ejecutar una query por sus efectos secundarios (validación, sondeo), usa **`PERFORM`**:

```sql
PERFORM 1 FROM dim_customer WHERE customer_id = 'ALFKI';
IF FOUND THEN
    RAISE NOTICE 'El cliente ALFKI existe';
END IF;
```

`PERFORM` es a `SELECT` lo que `PRINT` es a `RETURN`: ejecuta pero no entrega valor.

In [ ]:
%%sql
DO $$
BEGIN
    PERFORM 1 FROM northwind_dwh.dim_customer WHERE customer_id = 'ALFKI';
    
    IF FOUND THEN
        RAISE NOTICE 'Cliente ALFKI sí existe';
    ELSE
        RAISE WARNING 'Cliente ALFKI no encontrado';
    END IF;
END
$$;

## Bloques anidados y scope

Puedes anidar bloques `BEGIN ... END` dentro de otros. Cada bloque tiene su propio scope de variables:

```sql
DO $$
DECLARE
    x INTEGER := 10;
BEGIN
    DECLARE
        y INTEGER := 20;     -- y solo existe en el bloque interno
    BEGIN
        RAISE NOTICE 'x = %, y = %', x, y;
    END;
    -- aquí y ya no existe
END
$$;
```

Los bloques anidados son útiles principalmente para **capturar excepciones localmente** sin abortar el bloque externo (lo veremos en el siguiente bloque).

In [ ]:
%%sql
DO $$
DECLARE
    x INTEGER := 10;
BEGIN
    RAISE NOTICE 'Externo, antes del bloque interno: x = %', x;
    
    DECLARE
        x INTEGER := 100;    -- shadowing: misma variable, distinto scope
        y INTEGER := 999;
    BEGIN
        RAISE NOTICE 'Interno: x = %, y = %', x, y;
    END;
    
    RAISE NOTICE 'Externo, después del bloque interno: x = %', x;
END
$$;

**Punto pedagógico — shadowing:** el `x` del bloque interno **no es el mismo** que el externo. Es una variable nueva que tiene el mismo nombre. Al salir del bloque interno, la `x` externa sigue siendo 10. En general, evita shadowing en código real — confunde.

## Manejo básico de excepciones

El bloque `BEGIN ... END` puede tener una cláusula **`EXCEPTION`** que captura errores y permite recuperarse:

```sql
BEGIN
    -- código que puede fallar
EXCEPTION
    WHEN nombre_excepcion THEN
        -- manejo del error
    WHEN OTHERS THEN
        -- captura cualquier otra excepción
END;
```

Excepciones comunes en PostgreSQL:

| Excepción | Cuándo |
|---|---|
| `division_by_zero` | División entre cero |
| `unique_violation` | Choque con constraint UNIQUE/PK |
| `foreign_key_violation` | FK rota |
| `not_null_violation` | NULL en columna `NOT NULL` |
| `no_data_found` | `SELECT INTO STRICT` devolvió 0 filas |
| `too_many_rows` | `SELECT INTO STRICT` devolvió >1 fila |
| `OTHERS` | Comodín — cualquier otra excepción |

In [ ]:
%%sql
DO $$
DECLARE
    resultado NUMERIC;
BEGIN
    resultado := 10 / 0;
    RAISE NOTICE 'Resultado: %', resultado;     -- no se ejecuta
EXCEPTION
    WHEN division_by_zero THEN
        RAISE NOTICE 'Capturada: división entre cero';
    WHEN OTHERS THEN
        RAISE NOTICE 'Capturada otra excepción';
END
$$;

**Patrón típico — recuperar de un error sin abortar todo:**

Bloques anidados permiten que un sub-bloque falle sin abortar el externo:

In [ ]:
%%sql
DO $$
BEGIN
    RAISE NOTICE 'Paso 1';
    
    BEGIN
        PERFORM 1 / 0;
    EXCEPTION
        WHEN division_by_zero THEN
            RAISE NOTICE 'Paso 2 falló, lo manejé y sigo';
    END;
    
    RAISE NOTICE 'Paso 3 — el bloque externo continuó';
END
$$;

Lectura: el `EXCEPTION` solo aplica al bloque interno. El externo no sabe que algo falló — solo ve que el sub-bloque terminó, sea exitoso o por excepción capturada.

## Cierre

Lo que cubriste en este notebook:

| Tema | Construcción clave |
|---|---|
| Por qué PL/pgSQL | Lo que SQL puro **no puede** hacer: control de flujo, variables, manejo de excepciones |
| Bloque anónimo | `DO $$ BEGIN ... END $$;` |
| Variables | `DECLARE nombre TIPO := valor;` + asignación con `:=` |
| Mensajes | `RAISE NOTICE/WARNING/EXCEPTION '... %', valor;` |
| Asignar desde query | `SELECT col INTO variable FROM ...` (variantes `STRICT`) |
| Verificar resultado | Variable booleana `FOUND` después de la query |
| Ejecutar sin guardar | `PERFORM ...` en lugar de `SELECT` |
| Bloques anidados | Scope de variables + captura local de excepciones |
| Manejo de errores | `BEGIN ... EXCEPTION WHEN ... THEN ... END;` |

El siguiente notebook (**02 — Control de flujo**) profundiza en las construcciones procedurales: `IF`, `CASE`, `LOOP`, `FOR`, `WHILE`. Con eso ya podrás escribir bloques con lógica real.

---

<p align="center">
<a href="Readme.md">← Volver al índice del Tema 06</a> | <a href="02_control_de_flujo.ipynb">Siguiente: Notebook 02 — Control de flujo →</a>
</p>